# Gaussian Mixture Model (GMM) for TLS Communication Profiling and Anomaly Detection

Gaussian Mixture Models provide a **probabilistic density estimation framework** particularly suitable for modeling **multi-modal communication behavior**, which is typical in TLS traffic (CDNs, routing changes, application updates, device state shifts).
A GMM assumes that normal traffic is generated from a mixture of several Gaussian distributions.

Instead of assuming:

> “Normal traffic forms one cluster”

GMM assumes:

> “Normal traffic is composed of several behavior modes.”

Mathematically:

[
p(x) = \sum_{k=1}^{K} \pi_k , \mathcal{N}(x \mid \mu_k, \Sigma_k)
]

Where:

* (K) = number of components (behavior modes)
* (\pi_k) = mixture weights
* (\mu_k) = mean vector
* (\Sigma_k) = covariance matrix


TLS communication is rarely unimodal. These naturally form clusters in feature space. A single Gaussian is too restrictive.
A mixture captures this structure.


## What Is Being Modeled?

Input features may include:

* Flow duration
* Bytes sent / received
* Packet counts
* TLS version
* Cipher suite encoding
* JA3 / JA4 embeddings
* Inter-arrival timing statistics
* SNI categorical encodings
* Directional ratios

The GMM learns the **joint distribution** of these features for normal communication.

### GMM Implementation

**Covariance Structure Choices** are important. Scikit-learn provides options:

* `full` → full covariance matrix (most expressive)
* `tied` → shared covariance
* `diag` → diagonal only
* `spherical` → isotropic

For TLS:

* `full` is preferred if enough data
* `diag` if dimensionality is large

In high dimensions, full covariance may overfit.

**Selecting Number of Components K** is also a critical design decision.

Methods:

* BIC (Bayesian Information Criterion)
* AIC
* Cross-validation likelihood

In practice for TLS profiling:

* 3–10 components often sufficient per host
* Over-large K causes overfitting and weak anomaly contrast

In [1]:
from sklearn.mixture import GaussianMixture
import numpy as np

# Train on normal TLS flows
gmm = GaussianMixture(
    n_components=5,
    covariance_type="full",
    random_state=42
)
gmm.fit(X_train)

# Log-likelihood
log_likelihood = gmm.score_samples(X_test)

# Convert to anomaly score
anomaly_score = -log_likelihood

# Thresholding
threshold = np.percentile(-gmm.score_samples(X_train), 95)
anomalies = anomaly_score > threshold

NameError: name 'X_train' is not defined

### Anomaly Scoring

After training on normal data:

For a new flow (x):

Compute:

[
\text{AnomalyScore}(x) = - \log p(x)
]

Low probability → high anomaly score.

This gives a **continuous anomaly score**, which is ideal for:

* ROC evaluation
* Threshold-based detection
* Ranking suspicious flows


## Experiments

* Per-malware GMM
* Per-device GMM
* Per-OS GMM
* Per-application GMM

## Evaluation:

* AUROC
* TPR@1%FPR
* Per-family detection
* Likelihood histograms